In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
import base_tool

In [29]:
import pandas as pd
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

In [30]:
instrument_id = '511520'
trade_ymd = '20260319'
day = 5

In [ ]:
param_dict = {
    'name' : 'grid_trading',
    'threshold' : 0.002,
    'baseline' : baseline,
    'instrument_id' : instrument_id,
    'trade_ymd' : trade_ymd
}

In [ ]:
%%time
import logging
import os

class StrategyDemo():
    def __init__(self, param_dict={}) -> None:
        # self.model = joblib.load(file)
        
        self.__dict__.update(param_dict)
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}.pkl' 
        os.remove(data_file)

        self.position_last = 0

        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
        )
        self.logger = logging.getLogger(__name__)
        self.logger.info(f"策略初始化 - 短周期: {param_dict.get('short_window')}, 长周期: {param_dict.get('long_window')}, 阈值: {param_dict.get('threshold')}")

        self.price_list = deque(maxlen=self.long_window)
        self.prev_signal = 0

        return

    def on_snap(self, snap:dict) -> None:
        price = snap['price_last']

        if price == 0.0 or price == None:
            return

        self.price_list.append(price)

        if(len(self.price_list) < self.long_window):
            self.position_last = 0
            self.prev_signal = 0
            return

        short_ma = sum(list(self.price_list)[-self.short_window:]) / self.short_window
        long_ma = sum(self.price_list) / self.long_window
        diff = short_ma - long_ma

        if diff > self.threshold:
            current_signal = 1
        elif diff < -self.threshold:
            current_signal = -1
        else :
            current_signal = 0

        if current_signal != self.prev_signal:
            self.position_last = current_signal
            self.prev_signal = current_signal

            if current_signal == 1:
                self.logger.info(f"【买入信号】价格: {price}, 短均线: {short_ma:.4f}, 长均线: {long_ma:.4f}, 差值: {diff:.4f}")
            elif current_signal == -1:
                self.logger.info(f"【卖出信号】价格: {price}, 短均线: {short_ma:.4f}, 长均线: {long_ma:.4f}, 差值: {diff:.4f}")
            else:
                self.logger.info(f"【平仓/中性】价格: {price}, 均线回归中性区域")

        return

    

In [ ]:
%%time
def backtest_demo(instrument_id, trade_ymd, strategy_name):
    strategy = StrategyDemo(param_dict)
    snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
    position_dict = {}
    for snap in snap_list[:]:
        strategy.on_snap(snap)
        position_dict[snap['time_mark']] = strategy.position_last
    profit = base_tool.backtest_quick(instrument_id, trade_ymd, strategy_name, position_dict)
    return profit
backtest_demo('511520', '20260319', 'grid_trading')